# 4. Quadrant Quantization and Residual

## Objective
Split a grayscale image into four regions, quantize each region by rounding intensities down to multiples of 64, reassemble the image, and inspect the residual.

The residual is the original intensity minus the quantized intensity. It should stay in the range 0 to 63. Odd image dimensions are supported.

## Run

Install the packages listed in the repository-level requirements.txt. Set IMAGE_PATH to a local image path to replace the sample input, then run the code cell.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import data

# Set this to Path("path/to/image.jpg") to use your own image.
IMAGE_PATH = None

if IMAGE_PATH is None:
    image = data.camera()
else:
    image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

if image.shape[0] < 2 or image.shape[1] < 2:
    raise ValueError("Input image must be at least 2 x 2 pixels.")

def quantize(image: np.ndarray, step: int = 64) -> np.ndarray:
    """Round uint8 grayscale values down to a multiple of step."""
    return ((image.astype(np.uint16) // step) * step).astype(np.uint8)

middle_y, middle_x = image.shape[0] // 2, image.shape[1] // 2
quadrants = {
    "Top left": image[:middle_y, :middle_x],
    "Top right": image[:middle_y, middle_x:],
    "Bottom left": image[middle_y:, :middle_x],
    "Bottom right": image[middle_y:, middle_x:],
}
processed = {name: quantize(part) for name, part in quadrants.items()}

top = np.concatenate((processed["Top left"], processed["Top right"]), axis=1)
bottom = np.concatenate((processed["Bottom left"], processed["Bottom right"]), axis=1)
reconstructed = np.concatenate((top, bottom), axis=0)
residual = image.astype(np.int16) - reconstructed.astype(np.int16)

assert reconstructed.shape == image.shape
assert residual.min() >= 0 and residual.max() < 64

figure, axes = plt.subplots(4, 3, figsize=(15, 17))
for row, (name, part) in enumerate(quadrants.items()):
    quantized_part = processed[name]
    error_part = part.astype(np.int16) - quantized_part.astype(np.int16)
    panels = (
        (part, f"{name}: original", 255),
        (quantized_part, f"{name}: quantized", 255),
        (error_part, f"{name}: residual", 63),
    )
    for column, (panel, title, vmax) in enumerate(panels):
        axes[row, column].imshow(panel, cmap="gray", vmin=0, vmax=vmax)
        axes[row, column].set_title(title)
        axes[row, column].axis("off")

figure.tight_layout()
plt.show()

overview, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[1].imshow(reconstructed, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Reassembled quantized image")
axes[2].imshow(residual, cmap="gray", vmin=0, vmax=63)
axes[2].set_title("Quantization residual (0 to 63)")
for axis in axes:
    axis.axis("off")

overview.tight_layout()
plt.show()
print(f"Input shape: {image.shape}; reconstructed shape: {reconstructed.shape}")
print(f"Residual range: {residual.min()} to {residual.max()}; mean: {residual.mean():.2f}")